In [1]:
import os
import json
import torch
import warnings
import matplotlib.pyplot as plt
from IPython.display import display
from diffusers import FluxPipeline

warnings.filterwarnings("ignore")

# --- CONFIGURAZIONE BASE ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16
MODEL_ID = "black-forest-labs/FLUX.1-schnell"

print("Caricamento FLUX in memoria (potrebbe richiedere un minuto)...")
pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
pipe.set_progress_bar_config(disable=True)
print("✅ Modello caricato e pronto in VRAM!")

ModuleNotFoundError: No module named 'diffusers'

In [ ]:
def plot_attention_mask(A_target_tensor, title="Attrattore Semantico (A_target)"):
    """
    Prende il tensore [1, 4096, 1], lo "srotola" nella sua griglia spaziale 64x64
    e lo plotta come heatmap. Fondamentale per capire se il DB ha estratto 
    la forma giusta o solo spazzatura.
    """
    # Rimuoviamo le dimensioni extra e convertiamo in float32 per matplotlib
    mask_2d = A_target_tensor.squeeze().to(torch.float32).cpu().numpy()
    
    # Rimodelliamo sulla griglia latente (4096 -> 64x64)
    grid_size = int(mask_2d.shape[0] ** 0.5)
    mask_grid = mask_2d.reshape(grid_size, grid_size)
    
    plt.figure(figsize=(5, 5))
    plt.imshow(mask_grid, cmap='viridis', interpolation='nearest')
    plt.colorbar(label='Intensità Attenzione [0-1]')
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
# --- PARAMETRI DELL'ESPERIMENTO ---
DB_PATH = "data/dataset_v1/a_blue_sphere"  # Assicurati che esista A_target_sphere.pt
WORD_TO_ISOLATE = "sphere"
AMBIENT_PROMPT = "a crystal clear lake, hyperrealistic, 8k"
LAMBDA_VAL = 1.0
LAKE_SEED = 42  # Cambia questo per provare nuovi laghi

print(f"🔬 Avvio esperimento su: {AMBIENT_PROMPT}")

# =========================================================================
# 1. ESTRAZIONE DAL DB
# =========================================================================
mask_file = os.path.join(DB_PATH, f"A_target_{WORD_TO_ISOLATE}.pt")
if not os.path.exists(mask_file):
    print(f"⚠️ ATTENZIONE: {mask_file} non trovato. Devi prima lanciare lo script di compilazione del DB!")
else:
    A_target = torch.load(mask_file, map_location="cpu", weights_only=True).to(DEVICE, dtype=DTYPE)
    v0_db = torch.load(os.path.join(DB_PATH, "v0_velocity.pt"), map_location="cpu", weights_only=True).to(DEVICE, dtype=DTYPE)
    x0_db = torch.load(os.path.join(DB_PATH, "x0_noise.pt"), map_location="cpu", weights_only=True).to(DEVICE, dtype=DTYPE)
    
    # --- DEBUG VISIVO 1: Controllo della topologia ---
    plot_attention_mask(A_target, title=f"Maschera pre-calcolata: '{WORD_TO_ISOLATE}'")

    # =========================================================================
    # 2. IL MOSAICO QUANTISTICO
    # =========================================================================
    with torch.no_grad():
        ambient_embeds, ambient_pooled, ambient_txt_ids = pipe.encode_prompt(AMBIENT_PROMPT, prompt_2=None)
        
        generator = torch.Generator(device=DEVICE).manual_seed(LAKE_SEED)
        latents_lake, latent_image_ids = pipe.prepare_latents(
            1, pipe.transformer.config.in_channels // 4, 1024, 1024, DTYPE, DEVICE, generator
        )
        
        # Saldatura dei rumori
        latents = latents_lake * (1.0 - A_target) + x0_db * A_target

        # =========================================================================
        # 3. INTEGRAZIONE DIFFERENZIALE (ODE)
        # =========================================================================
        pipe.scheduler.set_timesteps(4, device=DEVICE)
        
        for i, t in enumerate(pipe.scheduler.timesteps):
            timestep_1d = (t / 1000.0).expand(latents.shape[0]).to(latents.dtype)

            # Corrente dell'ambiente
            v_ambient = pipe.transformer(
                hidden_states=latents, timestep=timestep_1d, guidance=None,
                pooled_projections=ambient_pooled, encoder_hidden_states=ambient_embeds,
                txt_ids=ambient_txt_ids, img_ids=latent_image_ids, return_dict=False
            )[0]
            
            # Iniezione della forza di struttura
            delta_v = A_target * (v0_db - v_ambient)
            v_stitch = v_ambient + (LAMBDA_VAL * delta_v)
            
            latents = pipe.scheduler.step(v_stitch, t, latents, return_dict=False)[0]

        # =========================================================================
        # 4. DECODIFICA E VISUALIZZAZIONE INLINE
        # =========================================================================
        latents = pipe._unpack_latents(latents, 1024, 1024, pipe.vae_scale_factor)
        latents = (latents / pipe.vae.config.scaling_factor) + pipe.vae.config.shift_factor
        
        image_tensor = pipe.vae.decode(latents, return_dict=False)[0]
        final_image = pipe.image_processor.postprocess(image_tensor, output_type="pil")[0]
        
        # --- DEBUG VISIVO 2: Risultato Finale ---
        print("\n🖼️ Risultato del Latent Stitching:")
        display(final_image)